# 02 — Direct edge-based QUBO validation

Builds the direct QUBO with one semantic binary variable per admissible service-edge pair. Flow equalities are penalized quadratically and active resource inequalities are encoded with binary slack variables.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c1
from oec_qaoa.edge_qubo import build_edge_qubo, solve_qubo_exact
from oec_qaoa.ilp import solve_reference_ilp

instance = canonical_c1()
reference = solve_reference_ilp(instance)
qubo = build_edge_qubo(instance)

print(f"Reference objective: {reference.objective:.3f}")
print(f"QUBO penalty: {qubo.penalty:.3f}")

## Service-specific preprocessing

Edges that cannot lie on a source-to-destination path for a service are removed before binary variables are introduced.

In [ ]:
summary = pd.DataFrame(
    {
        "quantity": [
            "semantic edge variables",
            "slack variables",
            "total logical variables",
            "quadratic couplings",
            "penalized equalities",
        ],
        "value": [
            qubo.semantic_count,
            len(qubo.variable_names) - qubo.semantic_count,
            len(qubo.variable_names),
            len(qubo.quadratic),
            len(qubo.constraints),
        ],
    }
)
summary

## Penalty construction

For C1 the unconstrained objective span is 7, hence the common penalty is

\[
\lambda = 8.
\]

Active inequalities are integer-scaled and written as

\[
\sum_j a_{rj}y_j+s_r=C_r,
\]

with powers-of-two slack encoding. Algebraically identical inequalities are included once.

In [ ]:
constraint_rows = []
for constraint in qubo.constraints:
    constraint_rows.append(
        {
            "constraint": constraint.name,
            "terms": len(constraint.coefficients),
            "rhs": constraint.rhs,
            "type": constraint.name.split(":")[0],
        }
    )

constraints = pd.DataFrame(constraint_rows)
constraints.groupby("type").agg(
    count=("constraint", "count"),
    mean_terms=("terms", "mean"),
    max_terms=("terms", "max"),
)

In [ ]:
slack_rows = []
for label, indices in qubo.slack_variables.items():
    slack_rows.append(
        {
            "constraint": label,
            "number_of_slack_bits": len(indices),
            "variables": ", ".join(
                qubo.variable_names[index] for index in indices
            ),
        }
    )

pd.DataFrame(slack_rows)

C1 contains the semantic edge variables together with the slack bits required by the active shared-resource inequalities.

## Exact QUBO solution

The QUBO minimum is certified classically by exact linearization of the quadratic products with McCormick constraints.

In [ ]:
qubo_solution = solve_qubo_exact(qubo)

print(f"QUBO solver success: {qubo_solution.success}")
print(f"QUBO minimum energy: {qubo_solution.energy:.3f}")
print(qubo_solution.message)

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "service": service.name,
            "reference_edges": reference.selected_edges[service.name],
            "qubo_edges": qubo_solution.selected_edges[service.name],
            "same_decoded_solution": (
                reference.selected_edges[service.name]
                == qubo_solution.selected_edges[service.name]
            ),
        }
        for service in instance.services
    ]
)
comparison

## Equivalence check

The QUBO energy, decoded service paths and processing placement are compared with the constrained C1 optimum.

In [ ]:
assert reference.success
assert qubo_solution.success
assert abs(qubo_solution.energy - reference.objective) < 1e-9
assert qubo_solution.selected_edges == reference.selected_edges

assert qubo_solution.selected_edges["h0"] == (
    "tx_S0_S1_t0_p0",
    "st_S1_t0_p0",
    "st_S1_t1_p1",
    "st_G_t2_p1",
    "proc_S1_t1",
    "tx_S1_G_t2_p1",
)

assert qubo_solution.selected_edges["h1"] == (
    "tx_S2_S1_t0_p0",
    "st_S1_t0_p0",
    "st_S1_t1_p0",
    "st_G_t2_p0",
    "tx_S1_G_t2_p0",
    "proc_G_t3",
)

print("C1 edge-QUBO equivalence: PASS")

## Result

The direct edge-QUBO reproduces the C1 constrained optimum and the decoded physical solution.